# MR604 — CIFAR-10 / CIFAR-100 grid (Gap 4)

Extends the four-metric benchmark beyond the MNIST family, which Section 5.17
names as the study's main threat to external validity and Section 6.3 designates
Gap 4 and the first extension.

**Backbone.** CIFAR uses the VGG-11 spec in `snnbench/backbones.py`, shared by
all three pathways so the paradigm comparison stays matched. The two-convolution
MNIST backbone reaches roughly 75 % on CIFAR-10, which would confound the
comparison with plain underfitting. MNIST-family code paths are untouched, so
every figure already in the thesis still comes from the code that produced it.

**What this predicts.** Section 5.17 commits to three falsifiable predictions
this grid tests:

1. the fully connected classifier remains the binding constraint under gate G1;
2. the direct-training accuracy advantage persists and widens on the harder task;
3. the crossover constant moves upward, because a denser input raises the
   synaptic operation count faster than the multiply-accumulate count it is
   compared against.

Report the outcome either way. A grid that fails every gate confirms the
thesis's central claim — that whether spiking saves energy is conditional and
measurable — rather than refuting it.

**Gate G3.** Colab is a *fourth* execution platform. Its latency figures are
reported separately and are never pooled with the RTX 5070, A100 or T4 numbers.

**Budget.** Roughly 40 h of A100 per dataset at 120 epochs, so about 80 h and
~440 compute units for both. Run cell by cell; everything resumes.

## 1. Check the GPU

Stop here if this is not an A100.

In [ ]:
!nvidia-smi

import subprocess, re, sys
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("\nAllocated:", out)
name = out.split(",")[0].strip() if out else ""
mem_mb = int(re.search(r"(\d+)", out.split(",")[1]).group(1)) if "," in out else 0

if "A100" in name:
    print("A100 — good. The full grid including T = 16 will fit.")
elif "L4" in name:
    print("L4 (24 GB). It will fit but runs roughly 3x slower than an A100 for "
          "the same compute-unit cost. Consider reconnecting for an A100.")
elif mem_mb and mem_mb < 20000:
    print(f"WARNING: {name} with {mem_mb} MB. BPTT through VGG-11 at T = 16 stores "
          "activations for every timestep and will almost certainly run out of memory. "
          "Change the runtime type to A100 before starting the grid.")
else:
    print(f"Unrecognised device: {name}. Check memory before running T = 16.")

Fri Sep 25 00:58:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   79C    P0             71W /   72W |    5248MiB /  23034MiB |     91%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Drive and locate the repository

Checkpoints and results go to Drive so a disconnect or the 24 h session cap
costs nothing: `train_one()` resumes from the last epoch. The dataset itself
stays on local disk, where it is faster and cheap to re-download.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib

DRIVE   = '/content/drive/MyDrive'
PROJECT = f'{DRIVE}/MR604'          # repo, checkpoints and results live here
REPO    = f'{PROJECT}/SNN-MR604'

os.makedirs(PROJECT, exist_ok=True)
print('project:', PROJECT)
print('repo exists:', os.path.isdir(REPO))
if not os.path.isdir(REPO):
    print('\nThe repository is not in Drive yet. Either:')
    print(f'  (a) upload the SNN-MR604 folder from your PC to {PROJECT}/, or')
    print('  (b) set GITHUB_URL in the next cell and clone it.')

Mounted at /content/drive
project: /content/drive/MyDrive/MR604
repo exists: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!while pgrep -f claude_run.sh > /dev/null; do sleep 60; done; echo runner finished

In [ ]:
# Optional: clone instead of uploading. Leave GITHUB_URL as None to skip.
GITHUB_URL = None    # e.g. 'https://github.com/dipag6/snn-four-metric-benchmark.git'

import os
if GITHUB_URL and not os.path.isdir(REPO):
    !git clone {GITHUB_URL} {REPO}
elif os.path.isdir(REPO):
    print('repo already present, nothing to clone')
else:
    raise SystemExit('No repo. Upload it to Drive or set GITHUB_URL above.')

%cd {REPO}
!ls

repo already present, nothing to clone
/content/drive/MyDrive/MR604/SNN-MR604
analyse.py		 MR604_Colab.ipynb     README.md	 run.py
CANONICAL.md		 MR604_Colab_v2.ipynb  requirements.txt  snnbench
COLAB_GUIDE.md		 patch_cifar.py        run_gpu.ps1	 tools
GPU_SETUP.md		 _pre_cifar_backup     run_gpu.sh
MR604_CIFAR_Colab.ipynb  __pycache__	       RUN_ON_5070.md


## 3. Dependencies

`spikingjelly` supplies the neuron models; `fvcore` is used only for the MNIST
operation-count cross-check and is optional here.

In [ ]:
!pip install -q spikingjelly fvcore
import torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__,
      '| cuda', torch.cuda.is_available())

torch 2.11.0+cu128 | torchvision 0.26.0+cu128 | cuda True


## 4. Verify the operation counts before spending any GPU time

This asserts that the derived MAC, parameter and spiking-site counts match the
pinned CIFAR figures, and prints the gate G1 break-even thresholds. The same
verification is what caught the harness defect reported in Section 4.5, so it is
run before training rather than after.

Expected: **153,031,680 MACs / 9,494,794 params / 152,074 sites** for CIFAR-10,
an ANN baseline of **703.946 µJ** per inference, against 19.509 µJ on MNIST.

In [ ]:
!python run.py verify --dataset cifar10
!python run.py verify --dataset cifar100

--- operation count verification ---
  conv1    MAC= 1,769,472  params=    1,792  sites=  65,536
  conv2    MAC=18,874,368  params=   73,856  sites=  32,768
  conv3    MAC=18,874,368  params=  295,168  sites=  16,384
  conv4    MAC=37,748,736  params=  590,080  sites=  16,384
  conv5    MAC=18,874,368  params=1,180,160  sites=   8,192
  conv6    MAC=37,748,736  params=2,359,808  sites=   8,192
  conv7    MAC= 9,437,184  params=2,359,808  sites=   2,048
  conv8    MAC= 9,437,184  params=2,359,808  sites=   2,048
  fc1      MAC=   262,144  params=  262,656  sites=     512
  fc2      MAC=     5,120  params=    5,130  sites=      10
  TOTAL    MAC=153,031,680  params=9,494,794  sites= 152,074
  fvcore unavailable or failed (fvcore cross-check is defined for the 1x28x28 backbone only); analytical count is authoritative
  verification: PASS

--- break-even thresholds (gate G1) ---
   T   s* @0.9pJ   s* @1.89pJ                    binding?
   2      2.5556       1.2169             neither bind

In [ ]:
# Independent check that the generic counter reproduces the published MNIST
# figures exactly. If this fails, no CIFAR number below should be believed.
from snnbench.backbones import mnist_spec_matches_legacy
mnist_spec_matches_legacy(verbose=True)
print('generic counter agrees with the thesis on MNIST -> CIFAR counts trustworthy')

  macs    generic=   4,241,152  thesis=   4,241,152  ok
  params  generic=     422,090  thesis=     422,090  ok
  sites   generic=      37,770  thesis=      37,770  ok
generic counter agrees with the thesis on MNIST -> CIFAR counts trustworthy


## 5. Smoke test — every configuration, two epochs

Proves the whole pipeline end to end: data, both encoders, BNTT at every T, the
N-layer threshold fold on conversion, the spike monitor, the gates and the result
JSON. Costs a few minutes. **Do not start the grid until this passes.**

In [ ]:
!python run.py smoke --dataset cifar10 \
    --out-dir {PROJECT}/results_smoke_cifar10 \
    --ckpt-dir {PROJECT}/ckpt_smoke_cifar10 \
    --data-root /content/data


7 configurations queued

--- operation count verification ---
  conv1    MAC= 1,769,472  params=    1,792  sites=  65,536
  conv2    MAC=18,874,368  params=   73,856  sites=  32,768
  conv3    MAC=18,874,368  params=  295,168  sites=  16,384
  conv4    MAC=37,748,736  params=  590,080  sites=  16,384
  conv5    MAC=18,874,368  params=1,180,160  sites=   8,192
  conv6    MAC=37,748,736  params=2,359,808  sites=   8,192
  conv7    MAC= 9,437,184  params=2,359,808  sites=   2,048
  conv8    MAC= 9,437,184  params=2,359,808  sites=   2,048
  fc1      MAC=   262,144  params=  262,656  sites=     512
  fc2      MAC=     5,120  params=    5,130  sites=      10
  TOTAL    MAC=153,031,680  params=9,494,794  sites= 152,074
  fvcore unavailable or failed (fvcore cross-check is defined for the 1x28x28 backbone only); analytical count is authoritative
  verification: PASS

### [1/7]
[ann_s42] result exists -- skipping (delete the json to force a rerun)

### [2/7]
[stbp_T4_s42] result exists -- ski

## 6. The CIFAR-10 grid

21 runs: 3 ANN baselines, 9 Path A (STBP, T ∈ {4, 8, 16}), 9 Path B (QCFS,
T ∈ {2, 4, 8}), three seeds each. Roughly 40 h on an A100.

Everything resumes: rerun this cell after any disconnect and finished runs are
skipped, while a partially trained run continues from its last checkpoint. The
loop below retries so that a dropped session restarts itself when you reconnect.

In [ ]:
RESULTS_10 = f'{PROJECT}/results_cifar10'
CKPT_10    = f'{PROJECT}/ckpt_cifar10'

!python run.py grid --dataset cifar10 --epochs 120 \
    --out-dir {RESULTS_10} --ckpt-dir {CKPT_10} --data-root /content/data


21 configurations queued

--- operation count verification ---
  conv1    MAC= 1,769,472  params=    1,792  sites=  65,536
  conv2    MAC=18,874,368  params=   73,856  sites=  32,768
  conv3    MAC=18,874,368  params=  295,168  sites=  16,384
  conv4    MAC=37,748,736  params=  590,080  sites=  16,384
  conv5    MAC=18,874,368  params=1,180,160  sites=   8,192
  conv6    MAC=37,748,736  params=2,359,808  sites=   8,192
  conv7    MAC= 9,437,184  params=2,359,808  sites=   2,048
  conv8    MAC= 9,437,184  params=2,359,808  sites=   2,048
  fc1      MAC=   262,144  params=  262,656  sites=     512
  fc2      MAC=     5,120  params=    5,130  sites=      10
  TOTAL    MAC=153,031,680  params=9,494,794  sites= 152,074
  fvcore unavailable or failed (fvcore cross-check is defined for the 1x28x28 backbone only); analytical count is authoritative
  verification: PASS

### [1/21]
[ann_s42] result exists -- skipping (delete the json to force a rerun)

### [2/21]
[ann_s123] result exists -- ski

In [ ]:
# Resume helper: keeps relaunching until every configuration has a result file.
# Safe to interrupt; safe to rerun.
import subprocess, time, glob

EXPECTED_RUNS = 21
for attempt in range(1, 13):
    done = len([f for f in glob.glob(f'{RESULTS_10}/*.json')
                if not f.split('/')[-1].startswith('_')])
    print(f'--- attempt {attempt}: {done}/{EXPECTED_RUNS} results present')
    if done >= EXPECTED_RUNS:
        print('grid complete')
        break
    r = subprocess.run(['python', 'run.py', 'grid', '--dataset', 'cifar10',
                        '--epochs', '120', '--out-dir', RESULTS_10,
                        '--ckpt-dir', CKPT_10, '--data-root', '/content/data'])
    if r.returncode != 0:
        print(f'exited {r.returncode}; retrying in 30 s')
        time.sleep(30)

## 7. Gates and the four-metric table — CIFAR-10

In [ ]:
!python run.py gates --dataset cifar10 --out-dir {RESULTS_10}
!python analyse.py --out-dir {RESULTS_10} || echo "(analyse.py flags may differ; see its --help)"

## 8. The CIFAR-100 grid

Same protocol, 100 classes. Only the output layer changes, so the backbone and
every constant carry over. Run this after CIFAR-10 has finished.

In [ ]:
RESULTS_100 = f'{PROJECT}/results_cifar100'
CKPT_100    = f'{PROJECT}/ckpt_cifar100'

!python run.py grid --dataset cifar100 --epochs 120 \
    --out-dir {RESULTS_100} --ckpt-dir {CKPT_100} --data-root /content/data

In [ ]:
!python run.py gates --dataset cifar100 --out-dir {RESULTS_100}

## 9. Read the result against the three predictions

Print the per-layer G1 outcome so prediction 1 can be checked directly: is the
fully connected classifier still the binding constraint, as it is on all four
MNIST-family datasets and all three platforms?

In [ ]:
import json, glob, os

for label, folder in [('CIFAR-10', RESULTS_10), ('CIFAR-100', RESULTS_100)]:
    files = sorted(f for f in glob.glob(f'{folder}/*.json')
                   if not os.path.basename(f).startswith('_'))
    if not files:
        print(f'{label}: no results yet'); continue
    print(f'\n=== {label}  ({len(files)} runs)')
    for f in files:
        r = json.load(open(f))
        acc = r.get('accuracy') or r.get('top1') or r.get('acc')
        rates = r.get('per_layer_spike_rate') or {}
        hottest = max(rates, key=rates.get) if rates else None
        print(f"  {os.path.basename(f)[:-5]:22s} acc={acc if acc is None else round(acc,2)!s:>7}"
              f"  hottest layer={hottest}"
              f"{'' if hottest is None else ' @ %.4f' % rates[hottest]}")

## 10. Back up to Drive

Results and checkpoints are already written to Drive by the paths above. This
cell only reports what is there, so you can confirm before closing the session.

In [ ]:
!du -sh {PROJECT}/* 2>/dev/null | sort -k2
print()
!ls -1 {RESULTS_10} 2>/dev/null | wc -l
!ls -1 {RESULTS_100} 2>/dev/null | wc -l